# GRPO DrugGPT GPCR Experiment

This notebook fine-tunes DrugGPT using GRPO-style reward feedback from a GPCR class classifier.


## 1) Environment and dependency bootstrap

Run the next cell once in the `.venv` kernel. It checks required packages and attempts optional installs.


In [2]:
import importlib
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REQUIRED = [
    "torch", "transformers", "datasets", "accelerate", "trl", "peft",
    "sklearn", "pandas", "numpy", "matplotlib", "seaborn", "tqdm", "xgboost"
]
OPTIONAL = ["rdkit", "vllm"]

UV_BIN = shutil.which("uv")

def in_virtual_env() -> bool:
    return sys.prefix != getattr(sys, "base_prefix", sys.prefix)

def build_install_cmd(package_name: str):
    if UV_BIN is not None:
        # Use uv-managed install path when uv is available.
        return [UV_BIN, "pip", "install", "--python", sys.executable, package_name], "uv pip"
    return [sys.executable, "-m", "pip", "install", package_name], "pip"

def ensure_pkg(import_name: str, pip_name: str | None = None, optional: bool = False):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        return True, "already installed"
    except Exception:
        if optional:
            return False, "optional and missing"
        cmd, installer = build_install_cmd(pip_name)
        try:
            subprocess.check_call(cmd)
            return True, f"installed via {installer}"
        except Exception as exc:
            return False, f"failed install via {installer}: {exc}"

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Virtual env active:", in_virtual_env())
print("uv detected:", UV_BIN is not None)
if UV_BIN:
    print("uv binary:", UV_BIN)

results = {}
for pkg in REQUIRED:
    ok, msg = ensure_pkg(pkg)
    results[pkg] = (ok, msg)

optional_map = {"rdkit": "rdkit", "vllm": "vllm"}
for pkg, pip_name in optional_map.items():
    ok, msg = ensure_pkg(pkg, pip_name=pip_name, optional=True)
    results[pkg] = (ok, msg)
    if (not ok) and pkg == "vllm":
        print("vLLM unavailable (common on Apple Silicon/MPS). Using transformers generation path.")

print("\nDependency status:")
for pkg, (ok, msg) in results.items():
    print(f"- {pkg}: {'OK' if ok else 'MISSING'} ({msg})")


Python: 3.12.11 (main, Aug  8 2025, 16:50:44) [Clang 20.1.4 ]
Executable: /Users/sahilkabir/Documents/School/DrugGenGPCR/.venv/bin/python
Platform: macOS-15.6.1-arm64-arm-64bit
Virtual env active: True
uv detected: True
uv binary: /Users/sahilkabir/.local/bin/uv

Dependency status:
- torch: OK (already installed)
- transformers: OK (already installed)
- datasets: OK (already installed)
- accelerate: OK (already installed)
- trl: OK (already installed)
- peft: OK (already installed)
- sklearn: OK (already installed)
- pandas: OK (already installed)
- numpy: OK (already installed)
- matplotlib: OK (already installed)
- seaborn: OK (already installed)
- tqdm: OK (already installed)
- xgboost: OK (already installed)
- rdkit: OK (already installed)
- vllm: OK (already installed)


In [3]:
import torch
import transformers
import trl
import xgboost

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("xgboost:", xgboost.__version__)


Device: mps
torch: 2.8.0
transformers: 5.5.4
trl: 1.1.0
xgboost: 3.2.0


## 2) Data loading and 80/20 split


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("gpcr_class_sequences_from_json.csv")
assert DATA_PATH.exists(), "Expected gpcr_class_sequences_from_json.csv in repo root"

df = pd.read_csv(DATA_PATH)
required_cols = {"sequence", "gpcr_binding_encoded"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["sequence", "gpcr_binding_encoded"]).copy()
df["gpcr_binding_encoded"] = df["gpcr_binding_encoded"].astype(int)
df["prompt"] = df["sequence"].map(lambda s: f"<|startoftext|><P>{s}<L>")
# remove classes with less than 3 sequences
df = df.groupby("gpcr_binding_encoded").filter(lambda x: len(x) >= 3)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["gpcr_binding_encoded"],
)

print("Total rows:", len(df))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Classes in train:", train_df["gpcr_binding_encoded"].nunique())
print("Classes in test:", test_df["gpcr_binding_encoded"].nunique())


Total rows: 1109
Train rows: 887
Test rows: 222
Classes in train: 78
Classes in test: 72


## 3) Classifier loading and stability guardrails


In [5]:
import os
import numpy as np
import xgboost as xgb

MODEL_PATH = Path("xgb_rdkit_classify_fp.json")
if not MODEL_PATH.exists():
    print("WARNING: xgb_rdkit_classify_fp.json not found in repo root.")
    print("Set MODEL_PATH to the correct location before training.")

xgb_model = xgb.XGBClassifier(
    nthread=1,
    predictor="cpu_predictor",
)

if MODEL_PATH.exists():
    xgb_model.load_model(str(MODEL_PATH))
    dummy = np.zeros((4, 2048), dtype=np.float32)
    _ = xgb_model.predict_proba(dummy)
    print("XGBoost model loaded and dummy inference passed.")


XGBoost model loaded and dummy inference passed.


## 4) Reward function integration (harsh invalid penalty)


In [6]:
from dataclasses import dataclass
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

INVALID_SMILES_PENALTY = -0.5 

@dataclass
class RewardResult:
    reward: float
    is_valid_smiles: bool
    pred_class: int
    confidence: float
    smiles: str

def smiles_to_fp(smiles: str, n_bits: int = 2048, radius: int = 4):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    bit_vect = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bit_vect, arr)
    return arr

def _first_token(s: str) -> str:
    s = s.strip()
    if not s:
        return ""
    parts = s.split()
    return parts[0] if parts else ""


def extract_smiles(decoded_text: str) -> str:
    """Extract candidate ligand SMILES.

    - If ``<L>`` is present (prompt + completion), use the substring after the first ``<L>``.
    - Otherwise treat the string as completion-only (matches TRL's decoded completion).
    """
    if not decoded_text:
        return ""
    if "<L>" in decoded_text:
        tail = decoded_text.split("<L>", 1)[1].strip()
        return _first_token(tail)
    return _first_token(decoded_text)


def reward_from_decoded(decoded_text: str, true_class: int):
    smiles = extract_smiles(decoded_text)
    fp = smiles_to_fp(smiles)
    if fp is None or (not MODEL_PATH.exists()):
        return RewardResult(
            reward=INVALID_SMILES_PENALTY,
            is_valid_smiles=False,
            pred_class=-1,
            confidence=0.0,
            smiles=smiles,
        )

    X = fp.reshape(1, -1)
    logits = xgb_model.predict(X, output_margin=True)[0]
    pred_class = int(np.argmax(logits))
    conf_true = float(logits[int(true_class)]) if int(true_class) < len(logits) else 0.0
    reward = logits[true_class] - max([logits[i] for i in range(len(logits)) if i != true_class])

    return RewardResult(
        reward=reward,
        is_valid_smiles=True,
        pred_class=pred_class,
        confidence=conf_true,
        smiles=smiles,
    )


## 5) DrugGPT setup and generation helpers


In [7]:
from transformers import AutoTokenizer, GPT2LMHeadModel, LogitsProcessor
from transformers.generation.logits_process import LogitsProcessorList

MODEL_NAME = "liyuesen/druggpt"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# bf16 is appropriate on CUDA; on MPS, sampling (softmax + multinomial) is much more stable in float32.
_policy_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=_policy_dtype).to(device)
model.train()


class ClampLogitsProcessor(LogitsProcessor):
    def __call__(self, input_ids, scores):
        s = torch.nan_to_num(scores, nan=0.0, posinf=50.0, neginf=-50.0)
        return torch.clamp(s, -50.0, 50.0)


def generate_decoded(prompt: str, max_new_tokens: int = 64):
    input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    prompt_len = int(input_ids.shape[1])
    attention_mask = input_ids.ne(tokenizer.pad_token_id).float()

    with torch.no_grad():
        out = model.generate(
            input_ids,
            do_sample=True,
            temperature=0.9,
            top_p=1.0,
            top_k=0,
            max_new_tokens=max_new_tokens,
            attention_mask=attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=LogitsProcessorList([ClampLogitsProcessor()]),
        )

    gen_ids = out[0, prompt_len:]
    completion = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return prompt + completion


Loading weights: 100%|██████████| 149/149 [00:00<00:00, 32977.22it/s]
GPT2LMHeadModel LOAD REPORT from: liyuesen/druggpt
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 6) GRPO trainer wiring (TRL) and fallback local loop

This notebook prefers TRL's `GRPOTrainer` when available. If unavailable in your installed TRL version, it falls back to a lightweight local policy-gradient style loop so you can still run the experiment.


In [7]:
from collections import defaultdict
from datetime import datetime
from datasets import Dataset
from transformers import LogitsProcessor
from transformers.generation.logits_process import LogitsProcessorList
from peft import LoraConfig, get_peft_model
from rdkit import RDLogger
import numpy as np
RDLogger.DisableLog('rdApp.*')

OUTPUT_DIR = Path("outputs/grpo_druggpt")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics = []
INVALID_TRACK = []

test_batch_size = 16

use_trl_grpo = True
trl_grpo_available = False
try:
    from trl import GRPOConfig, GRPOTrainer
    trl_grpo_available = True
except Exception as exc:
    print(f"TRL GRPO APIs unavailable in this environment: {exc}")

# GRPO + transformers sampling calls torch.multinomial on softmax probs; MPS often raises
# "probability tensor contains either inf, nan or element < 0" even when CUDA/CPU do not.
# Also set TrainingArguments.use_cpu so Accelerator does not keep using MPS after we move the model.

_grpo_use_cpu = False
"""
if use_trl_grpo and trl_grpo_available and device.type == "mps":
    print("Moving policy to CPU for GRPO (MPS multinomial / sampling stability). Training will be slower.")
    device = torch.device("cpu")
    model = model.to(device)
    _grpo_use_cpu = True
"""


if use_trl_grpo and trl_grpo_available:
    def trl_reward_fn(prompts, completions, **kwargs):
        labels = kwargs.get("gpcr_binding_encoded", [])
        log_metric = kwargs.get("log_metric")

        rewards = []
        valids = []
        confs = []

        for prompt, completion, label in zip(prompts, completions, labels, strict=True):
            full_text = prompt + completion
            rr = reward_from_decoded(full_text, int(label))

            rewards.append(rr.reward)
            valids.append(1.0 if rr.is_valid_smiles else 0.0)
            confs.append(rr.confidence)

        if log_metric is not None and valids:
            log_metric("invalid_rate", 1.0 - (sum(valids) / len(valids)))
            log_metric("confidence_mean", float(sum(confs) / len(confs)))

        rewards = np.array(rewards, dtype=np.float32)
        mean = rewards.mean()
        std = rewards.std()
        if std < 1e-6:
            normalized = rewards - mean
        else:
            normalized = (rewards - mean) / (std + 1e-8)

        return normalized.tolist()

    train_ds = Dataset.from_pandas(
        train_df[["prompt", "gpcr_binding_encoded"]].reset_index(drop=True)
    )

    class ClampLogitsProcessor(LogitsProcessor):
        """Stabilize sampling: clamp alone does not fix NaNs, which break softmax + multinomial."""

        def __call__(self, input_ids, scores):
            s = torch.nan_to_num(scores, nan=0.0, posinf=50.0, neginf=-50.0)
            return torch.clamp(s, -50.0, 50.0)

    grpo_config = GRPOConfig(
        output_dir=str(OUTPUT_DIR / "trl_grpo"),
        use_cpu=_grpo_use_cpu,
        bf16=(device.type == "cuda"),
        num_train_epochs=1,
        learning_rate=1e-8,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        # Keep generation settings divisible to satisfy TRL GRPO constraints.
        num_generations=12,
        generation_batch_size=12,
        logging_steps=5,
        save_steps=50,
        max_completion_length=64, # could lower to 64 ?
        report_to=[],
        # Sampling: use temperature for diversity. Avoid top_k + top_p together — on some setups (esp. MPS)
        # stacked filters can zero out all mass and make torch.multinomial fail with inf/nan probabilities.
        temperature=0.9,
        top_p=1.0,
        top_k=0,
        beta=0.02,
        warmup_ratio=0.1
    )
    grpo_config.logits_processor = LogitsProcessorList([ClampLogitsProcessor()])

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["c_attn", "c_proj", "c_fc"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    grpo_trainer = GRPOTrainer(
        model=model,
        reward_funcs=trl_reward_fn,
        args=grpo_config,
        train_dataset=train_ds,
        eval_dataset=test_df[["prompt", "gpcr_binding_encoded"]].reset_index(drop=True)
    )
    print("Starting TRL GRPOTrainer training...")
    trl_result = grpo_trainer.train()
    print("TRL GRPOTrainer finished. Building metrics rows for plotting.")

    # Populate metrics from TRL logs so plotting works even when fallback loop is skipped.
    trl_steps = []
    for row in grpo_trainer.state.log_history:
        if "step" not in row:
            continue
        step_val = int(row["step"])
        if step_val <= 0:
            continue
        trl_steps.append(step_val)
        metrics.append(
            {
                "step": step_val,
                "split": "train",
                "reward_mean": float(row.get("reward", np.nan)),
                "confidence_mean": float(row.get("confidence_mean", np.nan)),
                "invalid_rate": float(row.get("invalid_rate", np.nan)),
                "loss": float(row.get("loss", np.nan)),
            }
        )

    # Evaluate test set at each saved checkpoint for proper per-step metrics.
    from safetensors.torch import load_file as _load_safetensors
    from peft import set_peft_model_state_dict
    from tqdm.auto import tqdm

    ckpt_root = OUTPUT_DIR / "trl_grpo"
    ckpt_dirs = sorted(
        [p for p in ckpt_root.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[1]),
    )

    print(f"Evaluating test set across {len(ckpt_dirs)} checkpoints …")
    for ckpt_dir in tqdm(ckpt_dirs, desc="Checkpoint eval"):
        step_val = int(ckpt_dir.name.split("-")[1])
        adapter_path = ckpt_dir / "adapter_model.safetensors"
        if not adapter_path.exists():
            continue
        adapter_state = _load_safetensors(str(adapter_path), device=str(device))
        set_peft_model_state_dict(model, adapter_state)
        model.eval()

        t_rewards, t_confs, t_invalids = [], [], []
        for _, trow in test_df.iterrows():
            dec = generate_decoded(trow["prompt"])
            rr = reward_from_decoded(dec, int(trow["gpcr_binding_encoded"]))
            t_rewards.append(rr.reward)
            t_confs.append(rr.confidence)
            t_invalids.append(0 if rr.is_valid_smiles else 1)

        metrics.append({
            "step": step_val,
            "split": "test",
            "reward_mean": float(np.mean(t_rewards)) if t_rewards else np.nan,
            "confidence_mean": float(np.mean(t_confs)) if t_confs else np.nan,
            "invalid_rate": float(np.mean(t_invalids)) if t_invalids else np.nan,
            "loss": float(np.nan),
        })

    max_steps = grpo_config.max_steps

def sample_batch(frame, n):
    return frame.sample(n=min(n, len(frame)), replace=False, random_state=None)

def estimate_sequence_logprob(input_ids: torch.Tensor):
    # Teacher-forced score of generated continuation; simple surrogate objective.
    with torch.set_grad_enabled(True):
        outputs = model(input_ids=input_ids)
        logits = outputs.logits[:, :-1, :]
        labels = input_ids[:, 1:]
        log_probs = torch.log_softmax(logits, dim=-1)
        token_lp = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
        return token_lp.mean()

"""
max_steps = 100
train_batch_size = 8
test_eval_every = 10
test_batch_size = 16
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-6)

if (not use_trl_grpo) or (not trl_grpo_available):
    print("Running local surrogate GRPO-style loop (fallback path).")
for step in range(1, max_steps + 1):
    model.train()
    batch = sample_batch(train_df, train_batch_size)
    rewards = []
    confs = []
    invalids = []
    losses = []

    for _, row in batch.iterrows():
        prompt = row["prompt"]
        true_class = int(row["gpcr_binding_encoded"])
        decoded = generate_decoded(prompt)
        rr = reward_from_decoded(decoded, true_class)

        full_ids = torch.tensor(tokenizer.encode(decoded), dtype=torch.long).unsqueeze(0).to(device)
        seq_lp = estimate_sequence_logprob(full_ids)
        loss = -(rr.reward * seq_lp)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        rewards.append(rr.reward)
        confs.append(rr.confidence)
        invalids.append(0 if rr.is_valid_smiles else 1)
        losses.append(float(loss.detach().cpu().item()))

    train_reward = float(np.mean(rewards)) if rewards else 0.0
    train_conf = float(np.mean(confs)) if confs else 0.0
    train_invalid = float(np.mean(invalids)) if invalids else 0.0
    train_loss = float(np.mean(losses)) if losses else 0.0

    row_metrics = {
        "step": step,
        "split": "train",
        "reward_mean": train_reward,
        "confidence_mean": train_conf,
        "invalid_rate": train_invalid,
        "loss": train_loss,
    }
    metrics.append(row_metrics)

    if step % test_eval_every == 0:
        model.eval()
        tb = sample_batch(test_df, test_batch_size)
        t_rewards, t_confs, t_invalids = [], [], []
        for _, trow in tb.iterrows():
            dec = generate_decoded(trow["prompt"])
            rr = reward_from_decoded(dec, int(trow["gpcr_binding_encoded"]))
            t_rewards.append(rr.reward)
            t_confs.append(rr.confidence)
            t_invalids.append(0 if rr.is_valid_smiles else 1)

        metrics.append(
            {
                "step": step,
                "split": "test",
                "reward_mean": float(np.mean(t_rewards)) if t_rewards else 0.0,
                "confidence_mean": float(np.mean(t_confs)) if t_confs else 0.0,
                "invalid_rate": float(np.mean(t_invalids)) if t_invalids else 0.0,
                "loss": np.nan,
            }
        )

    if step % 20 == 0:
        ckpt_dir = OUTPUT_DIR / f"checkpoint_step_{step}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ckpt_dir)
        tokenizer.save_pretrained(ckpt_dir)

    if step % 5 == 0:
        print(f"step={step} train_reward={train_reward:.4f} train_invalid={train_invalid:.3f}")
"""

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/sahilkabir/Documents/School/DrugGenGPCR/.venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 0, 'bos_token_id': 53082, 'pad_token_id': 53081}.


Starting TRL GRPOTrainer training...


/Users/sahilkabir/Documents/School/DrugGenGPCR/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,0.036026
10,0.015587
15,0.145828
20,0.095789
25,0.476355
30,-0.031608
35,0.069362
40,-0.018094
45,0.232184
50,0.006104


: 

: 

In [ ]:
# Evaluate test set at each saved checkpoint for proper per-step metrics.
# This cell can be run standalone (without re-running training) since it
# loads each checkpoint as a fresh PeftModel on top of the base weights.
from peft import PeftModel
from tqdm.auto import tqdm
from rdkit import RDLogger
import numpy as np
RDLogger.DisableLog('rdApp.*')

OUTPUT_DIR = Path("outputs/grpo_druggpt")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metrics = []
INVALID_TRACK = []

ckpt_root = OUTPUT_DIR / "trl_grpo"
ckpt_dirs = sorted(
    [p for p in ckpt_root.glob("checkpoint-*") if p.is_dir()],
    key=lambda p: int(p.name.split("-")[1]),
)

base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=_policy_dtype).to(device)

print(f"Evaluating test set across {len(ckpt_dirs)} checkpoints …")
for ckpt_dir in tqdm(ckpt_dirs, desc="Checkpoint eval"):
    step_val = int(ckpt_dir.name.split("-")[1])
    adapter_path = ckpt_dir / "adapter_model.safetensors"
    if not adapter_path.exists():
        continue

    eval_model = PeftModel.from_pretrained(base_model, str(ckpt_dir))
    eval_model.eval()

    old_model = model
    model = eval_model

    t_rewards, t_confs, t_invalids = [], [], []
    for _, trow in test_df[:50].iterrows():
        dec = generate_decoded(trow["prompt"])
        rr = reward_from_decoded(dec, int(trow["gpcr_binding_encoded"]))
        t_rewards.append(rr.reward)
        t_confs.append(rr.confidence)
        t_invalids.append(0 if rr.is_valid_smiles else 1)

    model = old_model
    del eval_model

    metrics.append({
        "step": step_val,
        "split": "test",
        "reward_mean": float(np.mean(t_rewards)) if t_rewards else np.nan,
        "confidence_mean": float(np.mean(t_confs)) if t_confs else np.nan,
        "invalid_rate": float(np.mean(t_invalids)) if t_invalids else np.nan,
        "loss": float(np.nan),
    })
    print(f"step={step_val} test_reward={float(np.mean(t_rewards)):.4f} test_invalid={np.mean(t_invalids):.3f}")

del base_model


Loading weights: 100%|██████████| 149/149 [00:00<00:00, 38361.75it/s]
GPT2LMHeadModel LOAD REPORT from: liyuesen/druggpt
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating test set across 16 checkpoints …


Checkpoint eval:   6%|▋         | 1/16 [01:22<20:35, 82.36s/it]

step=50 test_reward=-0.8774 test_invalid=0.300


/Users/sahilkabir/Documents/School/DrugGenGPCR/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## 7) Save artifacts and visualize train/test improvements


In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

train_df[["uniprot_id"]].reset_index(drop=True).to_csv(OUTPUT_DIR / "train_ids.csv", index=False)
test_df[["uniprot_id"]].reset_index(drop=True).to_csv(OUTPUT_DIR / "test_ids.csv", index=False)

hparams = {
    "model_name": MODEL_NAME,
    "invalid_smiles_penalty": INVALID_SMILES_PENALTY,
    "max_steps": max_steps,
    "per_device_train_batch_size": grpo_config.per_device_train_batch_size,
    "gradient_accumulation_steps": grpo_config.gradient_accumulation_steps,
    "save_steps": grpo_config.save_steps,
    "test_eval_checkpoint_based": True,
    "timestamp": datetime.now().isoformat(),
    "device": str(device),
}
(OUTPUT_DIR / "hparams.json").write_text(json.dumps(hparams, indent=2))

required_plot_cols = {"step", "split", "reward_mean", "confidence_mean", "invalid_rate", "loss"}
missing_plot_cols = sorted(required_plot_cols - set(metrics_df.columns))
if metrics_df.empty or missing_plot_cols:
    print("No plottable metrics yet.")
    print("Run the GRPO training cell first so metrics contain train/test rows.")
    if missing_plot_cols:
        print("Missing columns:", missing_plot_cols)
    print("Available columns:", list(metrics_df.columns))
else:
    sns.set_theme(style="whitegrid")

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for split in ["train", "test"]:
        sub = metrics_df[metrics_df["split"] == split].copy()
        if sub.empty:
            continue
        sub["reward_smooth"] = sub["reward_mean"].rolling(window=5, min_periods=1).mean()
        sub["confidence_smooth"] = sub["confidence_mean"].rolling(window=5, min_periods=1).mean()
        sub["invalid_smooth"] = sub["invalid_rate"].rolling(window=5, min_periods=1).mean()
        axes[0, 0].plot(sub["step"], sub["reward_mean"], alpha=0.3, label=f"{split} raw")
        axes[0, 0].plot(sub["step"], sub["reward_smooth"], label=f"{split} smooth")
        axes[0, 1].plot(sub["step"], sub["confidence_smooth"], label=split)
        axes[1, 0].plot(sub["step"], sub["invalid_smooth"], label=split)

    axes[0, 0].set_title("Reward vs Step")
    axes[0, 1].set_title("Classifier Confidence vs Step")
    axes[1, 0].set_title("Invalid SMILES Rate vs Step")

    train_only = metrics_df[metrics_df["split"] == "train"]
    if not train_only.empty:
        axes[1, 1].plot(train_only["step"], train_only["loss"], label="train loss")
    axes[1, 1].set_title("Training Loss (surrogate)")

    for ax in axes.flatten():
        ax.legend(loc="best")
        ax.set_xlabel("Step")

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=200)
    plt.show()

In [ ]:
print("Reproducibility summary")
print("- model:", MODEL_NAME)
print("- train rows:", len(train_df))
print("- test rows:", len(test_df))
print("- invalid penalty:", INVALID_SMILES_PENALTY)
print("- output dir:", OUTPUT_DIR.resolve())
